In [2]:
import pandas as pd
import numpy as np

from ortools.sat.python import cp_model

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
decision_data = pd.read_csv(
    "../data/predictions/multi_objective_decision_scores.csv"
)

print("Rows:", len(decision_data))
print("Columns:", decision_data.columns.tolist())

Rows: 6
Columns: ['decision_rank', 'priority_rank', 'task_id', 'asset_id', 'section_id', 'department', 'maintenance_type', 'defect_type', 'severity', 'criticality', 'overdue_days', 'estimated_duration', 'required_manpower', 'status', 'failure_probability', 'urgency_score', 'priority_score', 'priority_category', 'predicted_delay_minutes', 'operational_impact_score', 'operational_impact_category', 'traffic_intensity', 'affected_trains_estimate', 'failure_risk_factor', 'urgency_factor', 'criticality_factor', 'overdue_factor', 'duration_factor', 'traffic_factor', 'operational_factor', 'maintenance_decision_score', 'decision_category']


In [4]:
weekly_plan = pd.read_csv(
    "../data/predictions/weekly_block_plan.csv"
)

print("Weekly scheduled tasks:", len(weekly_plan))


Weekly scheduled tasks: 4


In [5]:
weekly_plan.head()

,plan_sequence,task_id,section_id,department,start_slot,end_slot,start_time,end_time,estimated_duration,required_manpower,maintenance_decision_score,predicted_delay_minutes
0,1,SMMS001,MTJ-AGC-01,S&T,1,3,Day 1 00:30,Day 1 01:30,45,3,0.274500,0.0
1,2,TMS001,NDL-MTJ-01,ENGINEERING,1,4,Day 1 00:30,Day 1 02:00,90,8,0.322000,0.0
2,3,TDMS001,GWL-JHS-01,TRACTION,4,6,Day 1 02:00,Day 1 03:00,60,5,0.270667,0.0
3,4,TMS002,NDL-MTJ-02,ENGINEERING,4,6,Day 1 02:00,Day 1 03:00,60,5,0.224667,0.0


In [6]:
numeric_columns = [
    "estimated_duration",
    "required_manpower",
    "maintenance_decision_score",
    "predicted_delay_minutes",
    "overdue_days",
    "criticality"
]

for column in numeric_columns:
    if column in decision_data.columns:
        decision_data[column] = pd.to_numeric(
            decision_data[column],
            errors="coerce"
        )

decision_data[numeric_columns] = (
    decision_data[numeric_columns]
    .fillna(0)
)

print("Numeric data cleaned.")

Numeric data cleaned.


In [7]:
DAYS = 30
SLOT_MINUTES = 30

SLOTS_PER_DAY = (
    24 * 60 // SLOT_MINUTES
)

TOTAL_SLOTS = (
    DAYS * SLOTS_PER_DAY
)

print("Days:", DAYS)
print("Slots per day:", SLOTS_PER_DAY)
print("Total monthly slots:", TOTAL_SLOTS)

Days: 30
Slots per day: 48
Total monthly slots: 1440


In [8]:
monthly_block_windows = []

for day in range(DAYS):

    day_start = day * SLOTS_PER_DAY

    monthly_block_windows.append({
        "day": day,
        "block_id": f"D{day+1}_B1",
        "start_slot": day_start + 1,
        "end_slot": day_start + 6
    })

    monthly_block_windows.append({
        "day": day,
        "block_id": f"D{day+1}_B2",
        "start_slot": day_start + 7,
        "end_slot": day_start + 11
    })

print(
    "Monthly maintenance windows:",
    len(monthly_block_windows)
)

Monthly maintenance windows: 60


In [9]:
monthly_block_windows = []

for day in range(DAYS):

    day_start = day * SLOTS_PER_DAY

    monthly_block_windows.append({
        "day": day,
        "block_id": f"D{day+1}_B1",
        "start_slot": day_start + 1,
        "end_slot": day_start + 6
    })

    monthly_block_windows.append({
        "day": day,
        "block_id": f"D{day+1}_B2",
        "start_slot": day_start + 7,
        "end_slot": day_start + 11
    })

print(
    "Monthly maintenance windows:",
    len(monthly_block_windows)
)

Monthly maintenance windows: 60


In [10]:
available_slots = set()

for block in monthly_block_windows:

    for slot in range(
        block["start_slot"],
        block["end_slot"]
    ):
        available_slots.add(slot)

print(
    "Available monthly slots:",
    len(available_slots)
)

Available monthly slots: 270


In [11]:
MAX_TASKS = 200

candidate_tasks = (
    decision_data
    .sort_values(
        "maintenance_decision_score",
        ascending=False
    )
    .head(MAX_TASKS)
    .copy()
    .reset_index(drop=True)
)

print(
    "Monthly candidate tasks:",
    len(candidate_tasks)
)

Monthly candidate tasks: 6


In [12]:
candidate_tasks["duration_slots"] = np.ceil(
    candidate_tasks["estimated_duration"]
    / SLOT_MINUTES
).astype(int)

candidate_tasks["duration_slots"] = (
    candidate_tasks["duration_slots"]
    .clip(lower=1)
)

candidate_tasks[
    [
        "task_id",
        "estimated_duration",
        "duration_slots"
    ]
].head(10)

,task_id,estimated_duration,duration_slots
0,SMMS002,30,1
1,TDMS002,45,2
2,TMS001,90,3
3,SMMS001,45,2
4,TDMS001,60,2
5,TMS002,60,2


In [13]:
candidate_tasks["urgency_tier"] = np.select(
    [
        candidate_tasks["maintenance_decision_score"] >= 0.75,
        candidate_tasks["maintenance_decision_score"] >= 0.50,
        candidate_tasks["maintenance_decision_score"] >= 0.25
    ],
    [
        "CRITICAL",
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

candidate_tasks[
    [
        "task_id",
        "maintenance_decision_score",
        "urgency_tier"
    ]
].head(15)

,task_id,maintenance_decision_score,urgency_tier
0,SMMS002,0.362388,MEDIUM
1,TDMS002,0.361884,MEDIUM
2,TMS001,0.322000,MEDIUM
3,SMMS001,0.274500,MEDIUM
4,TDMS001,0.270667,MEDIUM
5,TMS002,0.224667,LOW


In [14]:
model = cp_model.CpModel()

print("Monthly CP-SAT model created.")

Monthly CP-SAT model created.


In [15]:
task_selected = {}
task_start = {}
task_end = {}

for i, row in candidate_tasks.iterrows():

    duration = int(
        row["duration_slots"]
    )

    task_selected[i] = model.NewBoolVar(
        f"task_selected_{i}"
    )

    task_start[i] = model.NewIntVar(
        0,
        TOTAL_SLOTS - duration,
        f"task_start_{i}"
    )

    task_end[i] = model.NewIntVar(
        0,
        TOTAL_SLOTS,
        f"task_end_{i}"
    )

    model.Add(
        task_end[i]
        ==
        task_start[i] + duration
    )

print(
    "Task variables created:",
    len(task_selected)
)

Task variables created: 6


In [16]:
for i, row in candidate_tasks.iterrows():

    duration = int(
        row["duration_slots"]
    )

    block_choices = []

    for j, block in enumerate(
        monthly_block_windows
    ):

        start_min = block["start_slot"]
        start_max = (
            block["end_slot"]
            - duration
        )

        if start_max >= start_min:

            choice = model.NewBoolVar(
                f"task_{i}_block_{j}"
            )

            block_choices.append(choice)

            model.Add(
                task_start[i] >= start_min
            ).OnlyEnforceIf(choice)

            model.Add(
                task_start[i] <= start_max
            ).OnlyEnforceIf(choice)

    model.Add(
        sum(block_choices)
        ==
        task_selected[i]
    )

print("Maintenance-window constraints added.")

Maintenance-window constraints added.


In [17]:
section_intervals = {}

for i, row in candidate_tasks.iterrows():

    duration = int(
        row["duration_slots"]
    )

    section = row["section_id"]

    interval = model.NewOptionalIntervalVar(
        task_start[i],
        duration,
        task_end[i],
        task_selected[i],
        f"section_interval_{i}"
    )

    section_intervals.setdefault(
        section,
        []
    ).append(interval)

In [18]:
for section, intervals in section_intervals.items():

    model.AddNoOverlap(intervals)

print("Section no-overlap constraints added.")

Section no-overlap constraints added.


In [19]:
MAX_MANPOWER = 12

In [20]:
manpower_intervals = []
manpower_demands = []

for i, row in candidate_tasks.iterrows():

    duration = int(
        row["duration_slots"]
    )

    manpower = int(
        row["required_manpower"]
    )

    interval = model.NewOptionalIntervalVar(
        task_start[i],
        duration,
        task_end[i],
        task_selected[i],
        f"manpower_interval_{i}"
    )

    manpower_intervals.append(interval)
    manpower_demands.append(manpower)

In [21]:
model.AddCumulative(
    manpower_intervals,
    manpower_demands,
    MAX_MANPOWER
)

print("Monthly manpower constraint added.")

Monthly manpower constraint added.


In [22]:
PERIODS = 5
DAYS_PER_PERIOD = 6

print(
    "Planning periods:",
    PERIODS
)

Planning periods: 5


In [23]:
task_period = {}

for i, row in candidate_tasks.iterrows():

    task_period[i] = model.NewIntVar(
        0,
        PERIODS - 1,
        f"task_period_{i}"
    )

In [24]:
for i, row in candidate_tasks.iterrows():

    # Convert starting slot to approximate period.
    period_size = (
        DAYS_PER_PERIOD
        * SLOTS_PER_DAY
    )

    for period in range(PERIODS):

        period_start = (
            period * period_size
        )

        period_end = (
            (period + 1)
            * period_size
            - 1
        )

        is_period = model.NewBoolVar(
            f"task_{i}_period_{period}"
        )

        model.Add(
            task_start[i] >= period_start
        ).OnlyEnforceIf(is_period)

        model.Add(
            task_start[i] <= period_end
        ).OnlyEnforceIf(is_period)

        model.Add(
            task_period[i] == period
        ).OnlyEnforceIf(is_period)

In [25]:
for i, row in candidate_tasks.iterrows():

    period_choices = []

    for period in range(PERIODS):

        choice = model.NewBoolVar(
            f"selected_{i}_period_{period}"
        )

        period_choices.append(choice)

    model.Add(
        sum(period_choices)
        ==
        task_selected[i]
    )

In [26]:
priority_weight = {
    "CRITICAL": 4,
    "HIGH": 3,
    "MEDIUM": 2,
    "LOW": 1
}

In [27]:
SCORE_SCALE = 1000
IMPACT_SCALE = 10

objective_terms = []

for i, row in candidate_tasks.iterrows():

    priority = int(
        row["maintenance_decision_score"]
        * SCORE_SCALE
    )

    impact = int(
        row["predicted_delay_minutes"]
        * IMPACT_SCALE
    )

    tier = row["urgency_tier"]

    weight = priority_weight[tier]

    coefficient = (
        priority * weight
        - impact
    )

    objective_terms.append(
        coefficient
        * task_selected[i]
    )

model.Maximize(
    sum(objective_terms)
)

print("Monthly optimization objective created.")

Monthly optimization objective created.


In [28]:
solver = cp_model.CpSolver()

solver.parameters.max_time_in_seconds = 60
solver.parameters.num_search_workers = 8

status = solver.Solve(model)

print(
    "Solver status:",
    solver.StatusName(status)
)

print(
    "Objective value:",
    solver.ObjectiveValue()
)

Solver status: OPTIMAL
Objective value: 1956.0


In [29]:
monthly_plan_rows = []

for i, row in candidate_tasks.iterrows():

    if solver.Value(
        task_selected[i]
    ) == 1:

        result = row.copy()

        result["start_slot"] = (
            solver.Value(
                task_start[i]
            )
        )

        result["end_slot"] = (
            solver.Value(
                task_end[i]
            )
        )

        monthly_plan_rows.append(
            result
        )

monthly_plan = pd.DataFrame(
    monthly_plan_rows
)

print(
    "Scheduled monthly tasks:",
    len(monthly_plan)
)

Scheduled monthly tasks: 4


In [30]:
def slot_to_datetime(slot):

    day = (
        slot // SLOTS_PER_DAY
    )

    slot_in_day = (
        slot % SLOTS_PER_DAY
    )

    total_minutes = (
        slot_in_day
        * SLOT_MINUTES
    )

    hours = (
        total_minutes // 60
    )

    minutes = (
        total_minutes % 60
    )

    return (
        f"Day {day + 1} "
        f"{hours:02d}:{minutes:02d}"
    )

In [31]:
monthly_plan["start_time"] = (
    monthly_plan["start_slot"]
    .apply(slot_to_datetime)
)

monthly_plan["end_time"] = (
    monthly_plan["end_slot"]
    .apply(slot_to_datetime)
)

In [32]:
monthly_plan["planning_period"] = (
    monthly_plan["start_slot"]
    // (DAYS_PER_PERIOD * SLOTS_PER_DAY)
) + 1

monthly_plan[
    [
        "task_id",
        "start_time",
        "end_time",
        "planning_period"
    ]
].head(15)

,task_id,start_time,end_time,planning_period
2,TMS001,Day 30 00:30,Day 30 02:00,5
3,SMMS001,Day 30 03:30,Day 30 04:30,5
4,TDMS001,Day 30 03:30,Day 30 04:30,5
5,TMS002,Day 30 04:30,Day 30 05:30,5


In [33]:
monthly_controller_plan = monthly_plan[
    [
        "task_id",
        "section_id",
        "department",
        "urgency_tier",
        "planning_period",
        "start_slot",
        "end_slot",
        "start_time",
        "end_time",
        "estimated_duration",
        "required_manpower",
        "maintenance_decision_score",
        "predicted_delay_minutes"
    ]
].copy()

monthly_controller_plan = (
    monthly_controller_plan
    .sort_values(
        ["start_slot", "section_id"]
    )
)

monthly_controller_plan.insert(
    0,
    "plan_sequence",
    range(
        1,
        len(monthly_controller_plan) + 1
    )
)

monthly_controller_plan.head(20)

,plan_sequence,task_id,section_id,department,urgency_tier,planning_period,start_slot,end_slot,start_time,end_time,estimated_duration,required_manpower,maintenance_decision_score,predicted_delay_minutes
2,1,TMS001,NDL-MTJ-01,ENGINEERING,MEDIUM,5,1393,1396,Day 30 00:30,Day 30 02:00,90,8,0.322000,0.0
4,2,TDMS001,GWL-JHS-01,TRACTION,MEDIUM,5,1399,1401,Day 30 03:30,Day 30 04:30,60,5,0.270667,0.0
3,3,SMMS001,MTJ-AGC-01,S&T,MEDIUM,5,1399,1401,Day 30 03:30,Day 30 04:30,45,3,0.274500,0.0
5,4,TMS002,NDL-MTJ-02,ENGINEERING,LOW,5,1401,1403,Day 30 04:30,Day 30 05:30,60,5,0.224667,0.0


In [34]:
monthly_controller_plan.to_csv(
    "../data/predictions/monthly_block_plan.csv",
    index=False
)

print(
    "Saved:",
    "../data/predictions/monthly_block_plan.csv"
)

Saved: ../data/predictions/monthly_block_plan.csv


In [35]:
monthly_summary = (
    monthly_plan
    .groupby("planning_period")
    .agg(
        tasks=("task_id", "count"),
        total_duration_minutes=(
            "estimated_duration",
            "sum"
        ),
        average_priority=(
            "maintenance_decision_score",
            "mean"
        ),
        total_manpower=(
            "required_manpower",
            "sum"
        )
    )
    .reset_index()
)

monthly_summary

,planning_period,tasks,total_duration_minutes,average_priority,total_manpower
0,5,4,255,0.272958,21


In [36]:
monthly_department_summary = (
    monthly_plan
    .groupby("department")
    .agg(
        tasks=("task_id", "count"),
        total_duration_minutes=(
            "estimated_duration",
            "sum"
        ),
        average_priority=(
            "maintenance_decision_score",
            "mean"
        )
    )
    .reset_index()
)

monthly_department_summary

,department,tasks,total_duration_minutes,average_priority
0,ENGINEERING,2,150,0.273333
1,S&T,1,45,0.274500
2,TRACTION,1,60,0.270667


In [37]:
critical_candidates = (
    candidate_tasks["urgency_tier"]
    == "CRITICAL"
).sum()

critical_scheduled = (
    monthly_plan["urgency_tier"]
    == "CRITICAL"
).sum()

if critical_candidates > 0:

    critical_coverage = (
        critical_scheduled
        / critical_candidates
    )

else:

    critical_coverage = 0

print(
    "Critical candidate tasks:",
    critical_candidates
)

print(
    "Critical scheduled tasks:",
    critical_scheduled
)

print(
    "Critical task coverage:",
    f"{critical_coverage:.2%}"
)

Critical candidate tasks: 0
Critical scheduled tasks: 0
Critical task coverage: 0.00%


In [38]:
conflict_count = 0

for section, group in monthly_plan.groupby(
    "section_id"
):

    group = group.sort_values(
        "start_slot"
    )

    previous_end = -1

    for _, row in group.iterrows():

        if (
            row["start_slot"]
            < previous_end
        ):
            conflict_count += 1

        previous_end = max(
            previous_end,
            row["end_slot"]
        )

print(
    "Section conflicts:",
    conflict_count
)

Section conflicts: 0


In [39]:
manpower_violations = 0

for slot in range(
    TOTAL_SLOTS
):

    active_manpower = 0

    for _, row in monthly_plan.iterrows():

        if (
            row["start_slot"]
            <= slot
            < row["end_slot"]
        ):

            active_manpower += int(
                row["required_manpower"]
            )

    if active_manpower > MAX_MANPOWER:

        manpower_violations += 1

print(
    "Manpower violations:",
    manpower_violations
)

Manpower violations: 0


In [40]:
candidate_count = len(
    candidate_tasks
)

scheduled_count = len(
    monthly_plan
)

monthly_coverage = (
    scheduled_count
    / candidate_count
)

print(
    "Candidate tasks:",
    candidate_count
)

print(
    "Scheduled tasks:",
    scheduled_count
)

print(
    "Monthly schedule coverage:",
    f"{monthly_coverage:.2%}"
)

Candidate tasks: 6
Scheduled tasks: 4
Monthly schedule coverage: 66.67%


In [41]:
print("===== MONTHLY PLAN SUMMARY =====")

print(
    "Candidate tasks:",
    len(candidate_tasks)
)

print(
    "Scheduled tasks:",
    len(monthly_plan)
)

print(
    "Schedule coverage:",
    f"{monthly_coverage:.2%}"
)

print(
    "Critical task coverage:",
    f"{critical_coverage:.2%}"
)

print(
    "Section conflicts:",
    conflict_count
)

print(
    "Manpower violations:",
    manpower_violations
)

print("\nMonthly workload:")
display(monthly_summary)

print("\nTop scheduled tasks:")
display(
    monthly_controller_plan.head(20)
)

===== MONTHLY PLAN SUMMARY =====
Candidate tasks: 6
Scheduled tasks: 4
Schedule coverage: 66.67%
Critical task coverage: 0.00%
Section conflicts: 0
Manpower violations: 0

Monthly workload:


,planning_period,tasks,total_duration_minutes,average_priority,total_manpower
0,5,4,255,0.272958,21



Top scheduled tasks:


,plan_sequence,task_id,section_id,department,urgency_tier,planning_period,start_slot,end_slot,start_time,end_time,estimated_duration,required_manpower,maintenance_decision_score,predicted_delay_minutes
2,1,TMS001,NDL-MTJ-01,ENGINEERING,MEDIUM,5,1393,1396,Day 30 00:30,Day 30 02:00,90,8,0.322000,0.0
4,2,TDMS001,GWL-JHS-01,TRACTION,MEDIUM,5,1399,1401,Day 30 03:30,Day 30 04:30,60,5,0.270667,0.0
3,3,SMMS001,MTJ-AGC-01,S&T,MEDIUM,5,1399,1401,Day 30 03:30,Day 30 04:30,45,3,0.274500,0.0
5,4,TMS002,NDL-MTJ-02,ENGINEERING,LOW,5,1401,1403,Day 30 04:30,Day 30 05:30,60,5,0.224667,0.0
